In [20]:
import pandas as pd


In [21]:
def get_data_file_path(filename) :
    BASE_DIR = './data'
    EXTENSION = '.csv'
    return f'{BASE_DIR}/{filename}{EXTENSION}'
cols = [
    '항구명', '호출부호', '년도','항차', '계선장소시설코드', '계선장소명', '선박종류','입항일자','입항목적','출항예정일시'
]

df = pd.read_csv(get_data_file_path('입항신고'), encoding='euc-kr', usecols=cols)

df.shape


(247056, 10)

In [22]:
call=df[df['항구명']=='부산']

call.shape
call.info()

# 중복행 확인
call.duplicated().sum()

<class 'pandas.DataFrame'>
Index: 57403 entries, 0 to 247052
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   항구명       57403 non-null  str  
 1   호출부호      57403 non-null  str  
 2   년도        57403 non-null  int64
 3   항차        57403 non-null  int64
 4   계선장소시설코드  57403 non-null  str  
 5   계선장소명     57403 non-null  str  
 6   선박종류      57403 non-null  str  
 7   입항일자      57403 non-null  str  
 8   입항목적      57403 non-null  str  
 9   출항예정일시    57403 non-null  str  
dtypes: int64(2), str(8)
memory usage: 4.8 MB


np.int64(2414)

In [23]:
# 중복행 제거
call = call.drop_duplicates().copy()


In [24]:
# 데이터 확인
call['입항'] = pd.to_datetime(call['입항일자'], errors='coerce')
call['입항'].isna().sum()
call['입항'].min()

# 이상치 제거
## 출항예정일시가 너무 뒤에 가있는거나 너무 앞에 가있는거는 아예 말이 안되는 데이터이기 때문에 지울것!

# call['입항'] = pd.to_datatime(call['입항일자'],errors='coerce')
planned_year = pd.to_numeric(call['출항예정일시'].str[:4], errors='coerce')

(planned_year > 2030).sum()
call.loc[planned_year > 2030, '출항예정일시']

10037     2424-05-20 16:00:00
12547     2424-05-25 18:00:00
83734     2824-09-30 13:00:00
88219     3024-10-11 05:30:00
95705     2204-10-25 04:00:00
112154    2054-11-25 12:00:00
177562    2205-03-24 18:00:00
191884    2525-04-21 18:00:00
194667    2525-04-17 16:10:00
Name: 출항예정일시, dtype: str

In [25]:

call['출항예정'] = pd.to_datetime(call['출항예정일시'].where(planned_year <= 2030), errors='coerce')

print(
call['입항'].min()
,call['출항예정'].min()
,call['입항'].max()
,call['출항예정'].max()
)




2004-01-01 00:00:00 2004-12-31 00:00:00 2025-07-31 15:00:00 2028-07-30 12:00:00


In [35]:
# 분석기간
# 24년5월1일 ~ 25년 7월 31일

(~(call['입항'].between('2024-05-01','2025-08-01',inclusive='left'))).sum()



call = call[call['입항'].between('2024-05-01','2025-08-01',inclusive='left')]


len(call)

53508

In [38]:
target = get_data_file_path('항만시설제원코드')

df2 = pd.read_csv(target,encoding='euc-kr')

facility = df2[df2['항만']=='부산항'].copy()

In [ ]:
facility['조인코드'] = facility['코드'].str.replace('-','',regex=False)

In [43]:
# 조인하기전에는 해당 데이터가 신회할 수 있는 데이터인지 확인할 것
facility[facility['조인코드'].duplicated(keep=False)]

facility = facility.drop_duplicates('조인코드')

facility[facility['조인코드'].duplicated(keep=False)]
# 계류시설 M
# 수역 W

,항만,시설이용구분,시설구분,시설명,선석 구분,코드,조인코드


In [58]:
calls = call.merge(
    facility[['조인코드','시설이용구분','시설구분']], # 여기에 넣고싶은 컬럼만 추가하면 해당 컬럼만 들어간다!!
    left_on='계선장소시설코드',
    right_on='조인코드',
    how='left'
)

In [47]:
calls

,항구명,호출부호,년도,항차,계선장소시설코드,계선장소명,선박종류,입항일자,입항목적,출항예정일시,입항,출항예정,조인코드,시설이용구분,시설구분
0,부산,9VFX4,2024,4,MBS05,신선대부두 5선석,풀컨테이너선,2024-05-01 03:00:00,양적하,2024-05-02 05:00:00,2024-05-01 03:00:00,2024-05-02 05:00:00,MBS05,계류시설 M,선석 B
1,부산,A8UX4,2024,3,MSN06,신항 2부두 3선석,풀컨테이너선,2024-05-01 04:15:00,양적하,2024-05-02 07:00:00,2024-05-01 04:15:00,2024-05-02 07:00:00,MSN06,계류시설 M,선석 S
2,부산,3FER7,2024,1,MSN02,신항 1부두 2선석,풀컨테이너선,2024-05-01 07:53:00,적하,2024-05-01 22:00:00,2024-05-01 07:53:00,2024-05-01 22:00:00,MSN02,계류시설 M,선석 S
3,부산,DSNC9,2024,17,MB604,자성대부두 64선석,풀컨테이너선,2024-05-01 07:50:00,양적하,2024-05-02 10:00:00,2024-05-01 07:50:00,2024-05-02 10:00:00,MB604,계류시설 M,선석 B
4,부산,D5WR4,2024,6,MBR04,감만부두 4선석,풀컨테이너선,2024-05-01 03:16:00,양적하,2024-05-01 15:00:00,2024-05-01 03:16:00,2024-05-01 15:00:00,MBR04,계류시설 M,선석 B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53503,부산,060333,2025,487,MK202,감천 2부두 2선석,기타선,2025-07-10 12:55:00,기타,2025-07-10 15:50:00,2025-07-10 12:55:00,2025-07-10 15:50:00,MK202,계류시설 M,선석 K
53504,부산,060333,2025,488,*WN01,남항 물량장,기타선,2025-07-10 16:25:00,기타,2025-07-11 07:50:00,2025-07-10 16:25:00,2025-07-11 07:50:00,*WN01,기타,기타
53505,부산,060333,2025,489,MQC01,한진 중공업 안벽,기타선,2025-07-11 08:15:00,기타,2025-07-11 11:00:00,2025-07-11 08:15:00,2025-07-11 11:00:00,MQC01,계류시설 M,안벽 Q
53506,부산,BL2167,2025,1,WAN03,남외항 N-3박지,원양 어선,2025-07-31 15:00:00,선용품적재,2025-07-31 16:00:00,2025-07-31 15:00:00,2025-07-31 16:00:00,WAN03,수역 W,박지 A


In [59]:
# 어떤 행동을 했을때에 나오는 데이터에 대해 어떻게 해결하려고 생각하지말고 왜 그렇게 나왔는지를 먼저 생각하자

unmatched = calls['시설이용구분'].isna()

calls.loc[unmatched,'계선장소명'].value_counts()


계선장소명
신항 7부두 2선석    291
신항 7부두 3선석    251
신항 7부두 1선석    250
북항 내 공사구간      11
Name: count, dtype: int64

In [62]:
in_new_7 = calls['계선장소시설코드'].str.startswith('MS7')

calls.loc[in_new_7,['시설이용구분','시설구분']] = ['계류시설 M','선석 S']


In [63]:

calls.loc[unmatched,'계선장소명'].value_counts()

계선장소명
신항 7부두 2선석    291
신항 7부두 3선석    251
신항 7부두 1선석    250
북항 내 공사구간      11
Name: count, dtype: int64

In [65]:
calls['시설이용구분']

# 계류시설을 정박지
# 수역시설을 묘박지

0        계류시설 M
1        계류시설 M
2        계류시설 M
3        계류시설 M
4        계류시설 M
          ...  
53503    계류시설 M
53504        기타
53505    계류시설 M
53506      수역 W
53507      수역 W
Name: 시설이용구분, Length: 53508, dtype: str

In [67]:
calls['시설구분'].value_counts()

시설구분
선석 B       12069
소형선부두 W    10672
선석 S        9197
박지 A        8491
기타          5962
선석 K        4872
조선소 Y        869
돌핀 D         595
안벽 Q         539
잔교 F         231
Name: count, dtype: int64

In [69]:
['선석 B','선석 S','선석 K']

'''
?????????????????????????????
출항예정일시에서 입항일시를 빼서 점유율을 본다?
'''

calls['예정체류'] = (calls['출항예정'] - calls['입항']).dt.total_seconds() / 3600

calls

,항구명,호출부호,년도,항차,계선장소시설코드,계선장소명,선박종류,입항일자,입항목적,출항예정일시,입항,출항예정,조인코드,시설이용구분,시설구분,예정체류
0,부산,9VFX4,2024,4,MBS05,신선대부두 5선석,풀컨테이너선,2024-05-01 03:00:00,양적하,2024-05-02 05:00:00,2024-05-01 03:00:00,2024-05-02 05:00:00,MBS05,계류시설 M,선석 B,26.000000
1,부산,A8UX4,2024,3,MSN06,신항 2부두 3선석,풀컨테이너선,2024-05-01 04:15:00,양적하,2024-05-02 07:00:00,2024-05-01 04:15:00,2024-05-02 07:00:00,MSN06,계류시설 M,선석 S,26.750000
2,부산,3FER7,2024,1,MSN02,신항 1부두 2선석,풀컨테이너선,2024-05-01 07:53:00,적하,2024-05-01 22:00:00,2024-05-01 07:53:00,2024-05-01 22:00:00,MSN02,계류시설 M,선석 S,14.116667
3,부산,DSNC9,2024,17,MB604,자성대부두 64선석,풀컨테이너선,2024-05-01 07:50:00,양적하,2024-05-02 10:00:00,2024-05-01 07:50:00,2024-05-02 10:00:00,MB604,계류시설 M,선석 B,26.166667
4,부산,D5WR4,2024,6,MBR04,감만부두 4선석,풀컨테이너선,2024-05-01 03:16:00,양적하,2024-05-01 15:00:00,2024-05-01 03:16:00,2024-05-01 15:00:00,MBR04,계류시설 M,선석 B,11.733333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53503,부산,060333,2025,487,MK202,감천 2부두 2선석,기타선,2025-07-10 12:55:00,기타,2025-07-10 15:50:00,2025-07-10 12:55:00,2025-07-10 15:50:00,MK202,계류시설 M,선석 K,2.916667
53504,부산,060333,2025,488,*WN01,남항 물량장,기타선,2025-07-10 16:25:00,기타,2025-07-11 07:50:00,2025-07-10 16:25:00,2025-07-11 07:50:00,*WN01,기타,기타,15.416667
53505,부산,060333,2025,489,MQC01,한진 중공업 안벽,기타선,2025-07-11 08:15:00,기타,2025-07-11 11:00:00,2025-07-11 08:15:00,2025-07-11 11:00:00,MQC01,계류시설 M,안벽 Q,2.750000
53506,부산,BL2167,2025,1,WAN03,남외항 N-3박지,원양 어선,2025-07-31 15:00:00,선용품적재,2025-07-31 16:00:00,2025-07-31 15:00:00,2025-07-31 16:00:00,WAN03,수역 W,박지 A,1.000000


In [75]:
berth_type = ['선석 B','선석 S','선석 K']

calls['시설구분'].isin(berth_type)

berth_calls = calls[calls['시설구분'].isin(berth_type)].copy()

len(berth_calls)

26138

In [76]:

# 예정체류시간안에 들어가는 친구들만 볼것!

berth_calls['예정체류'].isna().sum()

valid = berth_calls['예정체류'].between(0.5,720,inclusive='both')

berth_calls = berth_calls[valid].copy()

In [78]:
berth_calls.groupby('시설구분')['예정체류'].median().round(1)

시설구분
선석 B    16.0
선석 K    33.3
선석 S    20.8
Name: 예정체류, dtype: float64

In [91]:
period_start, period_end = pd.Timestamp('2024-05-01'), pd.Timestamp('2025-08-01')

period_hours = (period_end-period_start).total_seconds() / 3600

period_hours / 24


457.0

In [92]:
def occupied_hours(start,ends):
    #체류구간들을 시작시각 순으로 이어붙인다음에 겹침없이 선박이 있던 총 시간을 구함.
    interval = sorted(zip(start,ends))
    total = pd.Timedelta(0)
    current_start, current_end = interval[0]
    for start, end in interval[1]:
        if start<=current_start:
            current_end = max(current_end, end)
        else:
            total+=current_end-current_start
            current_start, current_end = start, ends
    total += current_end - current_start
    return total.total_seconds()/3600

In [95]:
rows = []

for code, group in berth_calls.groupby('계선장소시설코드') :
    rows.append({
        'code':code,
        'feri':group['개선장소명'].mode()[0],
        'gubun':group['시설구분'].iloc[0],
        'count':len(group),
        'percentage': occupied_hours(group['입항'],group['출항예정'])/period_hours * 100,
        'avg_ship':group['예정체류'].sum() / period_hours
    })

berth_index = pd.DataFrame(rows).set_index('code')


(128, 5)

In [ ]:
rows = []

for code, group in berth_calls.groupby('계선장소시설코드'):
    rows.append({
        '코드': code,
        '선석': group['계선장소명'].mode()[0],
        '시설구분': group['시설구분'].iloc[0],
        '입항건수': len(group),
        '점유율': occupied_hours(group['입항'], group['출항예정']) / period_hours * 100,
        '평균동시척수': group['예정체류'].sum() / period_hours,
    })

berth_index = pd.DataFrame(rows).set_index('코드')
berth_index.shape

In [ ]:
def occupied_hours(starts, ends):
    # 체류구간들을 시작 시각 순으로 이어 붙인다음에 겹침없이 선박이 있던 총 시간을 구함
    interval = sorted(zip(starts, ends))
    total = pd.Timedelta(0) # 그냥 여기는 새로운 시간구간 객체 설정
    current_start, current_end = interval[0]
    for start, end in interval[1:]: # 현재 계선장소를 기준으로 groupby를 했기 때문에 여러건수가 있음, 따라서 상단 주석에 작성한대로 각각의 체류구간들을 이어붙이는 작업을 진행함
        if start <= current_end: # 겹치는 구간이 있을 경우 해당 구간을 합치는 작업
            current_end = max(current_end, end)
        else: # 없으면 이제까지 합친 구간을 total에 넣고 새로 설정
            total += current_end - current_start
            current_start, current_end = start, end
    total += current_end - current_start
    return total.total_seconds() / 3600

In [106]:
berth_index.sort_values('점유율', ascending=False).head(6).round(2)

,선석,시설구분,입항건수,점유율,평균동시척수
코드,,,,,
MK304,감천 3부두 4선석,선석 K,130,92.21,2.83
MK201,감천 2부두 1선석,선석 K,327,90.89,2.15
MBR03,감만부두 3선석,선석 B,994,90.34,1.68
MBR04,감만부두 4선석,선석 B,907,89.03,1.58
MK303,감천 3부두 3선석,선석 K,108,88.94,2.01
MK405,감천 4부두 5선석,선석 K,433,88.26,2.22


In [107]:

buck_port = berth_index[berth_index['시설구분'] == '선석 B'].sort_values('선석')
buck_port

,선석,시설구분,입항건수,점유율,평균동시척수
코드,,,,,
MB501,5부두 51선석(양곡부두),선석 B,16,3.402322,0.034023
MB502,5부두 52선석(양곡부두),선석 B,28,35.798687,0.424497
MB701,7부두 71선석,선석 B,144,46.406364,0.554297
MB702,7부두 72선석,선석 B,188,31.573973,0.339141
MB703,7부두 73선석,선석 B,295,51.734288,0.602351
...,...,...,...,...,...
MB602,자성대부두 62선석,선석 B,134,30.706905,0.366686
MB603,자성대부두 63선석,선석 B,301,44.764770,0.592316
MB604,자성대부두 64선석,선석 B,317,38.860017,0.580483


In [108]:

new_port = berth_index[berth_index['시설구분'] == '선석 S'].sort_values('선석')
new_port

,선석,시설구분,입항건수,점유율,평균동시척수
코드,,,,,
MSN01,신항 1부두 1선석,선석 S,380,76.609835,0.843297
MSN02,신항 1부두 2선석,선석 S,386,84.328805,1.037483
MSN03,신항 1부두 3선석,선석 S,297,84.596098,0.976410
MSN04,신항 2부두 1선석,선석 S,464,78.932197,0.996976
MSN05,신항 2부두 2선석,선석 S,441,81.735807,0.953044
MSN06,신항 2부두 3선석,선석 S,413,85.546742,1.038258
MSN07,신항 2부두 4선석,선석 S,385,85.331419,1.038299
MSN08,신항 2부두 5선석,선석 S,361,81.625182,0.949777
MS301,신항 3부두 1선석,선석 S,292,56.476264,0.650821


In [104]:
berth_calls.groupby('계선장소시설코드').get_group('MSN05')


,항구명,호출부호,년도,항차,계선장소시설코드,계선장소명,선박종류,입항일자,입항목적,출항예정일시,입항,출항예정,조인코드,시설이용구분,시설구분,예정체류
17,부산,D5JH6,2024,2,MSN05,신항 2부두 2선석,풀컨테이너선,2024-05-01 20:05:00,양적하,2024-05-03 12:00:00,2024-05-01 20:05:00,2024-05-03 12:00:00,MSN05,계류시설 M,선석 S,39.916667
179,부산,9HA3184,2024,6,MSN05,신항 2부두 2선석,풀컨테이너선,2024-05-04 00:05:00,양적하,2024-05-05 21:00:00,2024-05-04 00:05:00,2024-05-05 21:00:00,MSN05,계류시설 M,선석 S,44.916667
251,부산,CQ2136,2024,3,MSN05,신항 2부두 2선석,풀컨테이너선,2024-05-04 23:00:00,양적하,2024-05-05 20:00:00,2024-05-04 23:00:00,2024-05-05 20:00:00,MSN05,계류시설 M,선석 S,21.000000
259,부산,CQEE7,2024,2,MSN05,신항 2부두 2선석,풀컨테이너선,2024-05-05 20:15:00,양적하,2024-05-07 03:00:00,2024-05-05 20:15:00,2024-05-07 03:00:00,MSN05,계류시설 M,선석 S,30.750000
328,부산,C6GG9,2024,1,MSN05,신항 2부두 2선석,풀컨테이너선,2024-05-07 04:05:00,양적하,2024-05-07 16:00:00,2024-05-07 04:05:00,2024-05-07 16:00:00,MSN05,계류시설 M,선석 S,11.916667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52796,부산,5LJR5,2025,4,MSN05,신항 2부두 2선석,LNG 운반선,2025-07-23 01:30:00,기타,2025-07-23 18:00:00,2025-07-23 01:30:00,2025-07-23 18:00:00,MSN05,계류시설 M,선석 S,16.500000
52806,부산,091321,2025,45,MSN05,신항 2부두 2선석,석유제품 운반선,2025-07-24 16:40:00,양하,2025-07-24 18:55:00,2025-07-24 16:40:00,2025-07-24 18:55:00,MSN05,계류시설 M,선석 S,2.250000
52907,부산,D5NX2,2025,3,MSN05,신항 2부두 2선석,풀컨테이너선,2025-07-27 02:00:00,양적하,2025-07-28 16:00:00,2025-07-27 02:00:00,2025-07-28 16:00:00,MSN05,계류시설 M,선석 S,38.000000
53097,부산,5LLT4,2025,2,MSN05,신항 2부두 2선석,풀컨테이너선,2025-07-28 15:44:00,양적하,2025-07-30 08:00:00,2025-07-28 15:44:00,2025-07-30 08:00:00,MSN05,계류시설 M,선석 S,40.266667
